# Generate photometry reference data

This notebook runs the nwb conversion pipeline as of commit `1c415f98146fea2f0f9e2d9bf39c442569048fa3` (before the photometry refactor) on real data and saves intermediate arrays at each processing step. These are used as ground-truth reference data for unit tests of the refactored photometry pipeline.

**Prerequisites:** Run `python tests/download_test_data.py` first to download test data from Box.

**Output:** `.npz` files in `tests/test_data/downloaded/reference/<commit-hash>/`

In [4]:
import struct
import numpy as np
import pandas as pd
import json
import os
import scipy.io
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from scipy.signal import butter, lfilter, hilbert, filtfilt
from scipy.sparse import diags, eye, csc_matrix
from scipy.sparse.linalg import spsolve
from sklearn.linear_model import Lasso
import subprocess

In [5]:
# We are generating reference data based on the state of the nwb conversion pipeline as of this commit:
# "Make impedance channel name check work with ports A/D (#191)" merged on 2/17/25 (most recent change to main)
# Technically, the last edit to photometry conversion was 9cc473dd27f9e40410ae84770a9af1bc29adc27d
# "Add support for recording and processing dLight with pyPhotometry (#174)" merged on 10/15/25
# We need reference data generated from the pipeline as of this commit because we
# want to make sure my big refactor (May 2026) produces output equivalent to the old code at each step.
# We'll save the reference data generated in this notebook in a folder named based on the commit hash as a way
# to keep track of things (especially useful should we want to do the same thing again for future refactors)
COMMIT_TAG = "1c415f98146fea2f0f9e2d9bf39c442569048fa3"

NOTEBOOK_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "tests").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name == "tests":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Output under downloaded/ so download_test_data.py keeps it in sync with Box
TEST_DATA_DIR = PROJECT_ROOT / "tests" / "test_data" / "downloaded"
OUTPUT_DIR = TEST_DATA_DIR / "reference" / COMMIT_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Test data dir: {TEST_DATA_DIR}")
print(f"Reference output dir: {OUTPUT_DIR}")

Test data dir: /Users/steph/berkelab/jdb_to_nwb/tests/test_data/downloaded
Reference output dir: /Users/steph/berkelab/jdb_to_nwb/tests/test_data/downloaded/reference/1c415f98146fea2f0f9e2d9bf39c442569048fa3


## Old pipeline functions
copied directly from `1c415f98146fea2f0f9e2d9bf39c442569048fa3`

In [6]:
def read_phot_data(phot_file_path):
    """Parse .phot file from Labview into a dict"""
    phot = {}
    with open(phot_file_path, "rb") as fid:
        phot["magic_key"] = struct.unpack(">I", fid.read(4))[0]
        phot["header_size"] = struct.unpack(">h", fid.read(2))[0]
        phot["main_version"] = struct.unpack(">h", fid.read(2))[0]
        phot["secondary_version"] = struct.unpack(">h", fid.read(2))[0]
        phot["sampling_rate"] = struct.unpack(">h", fid.read(2))[0]
        phot["bytes_per_sample"] = struct.unpack(">h", fid.read(2))[0]
        phot["num_channels"] = struct.unpack(">h", fid.read(2))[0]
        phot["file_name"] = fid.read(256).decode("utf-8").strip("\x00")
        phot["date"] = fid.read(256).decode("utf-8").strip("\x00")
        phot["time"] = fid.read(256).decode("utf-8").strip("\x00")
        phot["channels"] = []
        for i in range(4):
            phot["channels"].append({})
        for i in range(4):
            phot["channels"][i]["location"] = fid.read(256).decode("utf-8", errors="ignore").strip("\x00")
        for i in range(4):
            phot["channels"][i]["signal"] = fid.read(256).decode("utf-8", errors="ignore").strip("\x00")
        for i in range(4):
            phot["channels"][i]["freq"] = struct.unpack(">h", fid.read(2))[0]
        for i in range(4):
            phot["channels"][i]["max_v"] = struct.unpack(">h", fid.read(2))[0] / 32767.0
        for i in range(4):
            phot["channels"][i]["min_v"] = struct.unpack(">h", fid.read(2))[0] / 32767.0
        phot["signal_label"] = []
        for signal in range(8):
            signal_label = fid.read(256).decode("utf-8").strip("\x00")
            phot["signal_label"].append(signal_label)
        position = fid.tell()
        pad_size = phot["header_size"] - position
        phot["pad"] = fid.read(pad_size)
        data = np.fromfile(fid, dtype=np.dtype(">i2"))
        phot["data"] = np.reshape(data, (phot["num_channels"], -1), order="F")
    return phot


def read_box_data(box_file_path):
    """Parse .box file from Labview into a dict"""
    box = {}
    with open(box_file_path, "rb") as fid:
        box["magic_key"] = struct.unpack(">I", fid.read(4))[0]
        box["header_size"] = struct.unpack(">h", fid.read(2))[0]
        box["main_version"] = struct.unpack(">h", fid.read(2))[0]
        box["secondary_version"] = struct.unpack(">h", fid.read(2))[0]
        box["sampling_rate"] = struct.unpack(">h", fid.read(2))[0]
        box["bytes_per_sample"] = struct.unpack(">h", fid.read(2))[0]
        box["num_channels"] = struct.unpack(">h", fid.read(2))[0]
        box["file_name"] = fid.read(256).decode("utf-8").strip("\x00")
        box["date"] = fid.read(256).decode("utf-8").strip("\x00")
        box["time"] = fid.read(256).decode("utf-8").strip("\x00")
        box["ch1_location"] = fid.read(256).decode("utf-8").strip("\x00")
        box["ch2_location"] = fid.read(256).decode("utf-8").strip("\x00")
        box["ch3_location"] = fid.read(256).decode("utf-8").strip("\x00")
        position = fid.tell()
        pad_size = box["header_size"] - position
        box["pad"] = fid.read(pad_size)
        data = np.fromfile(fid, dtype=np.uint8)
        box["data"] = np.reshape(data, (box["num_channels"], -1), order="F")
    return box


def process_pulses(box):
    """Extract port visit times from the box dict and return visit timestamps in 10kHz sample time"""
    diff_data = np.diff(box["data"][2, :].astype(np.int16))
    start = np.where(diff_data < -1)[0][0]
    pulses = np.where(diff_data > 1)[0]
    visits = pulses - start
    return visits, start


def lockin_detection(input_signal, exc1, exc2, Fs, tau=10, filter_order=5, detrend=False, full=True):
    tau /= 1000  # Convert to seconds
    Fc = 1 / (2 * np.pi * tau)
    fL = 0.01
    b, a = butter(filter_order, Fc / (Fs / 2), "high")
    input_signal = lfilter(b, a, input_signal)
    demod1 = input_signal * exc1
    demod2 = input_signal * exc2
    if detrend:
        b, a = butter(filter_order, [fL, Fc] / (Fs / 2))
    else:
        b, a = butter(filter_order, Fc / (Fs / 2))
    if not full:
        sig1 = lfilter(b, a, demod1)
        sig2 = lfilter(b, a, demod2)
    else:
        sig1x = lfilter(b, a, demod1)
        sig2x = lfilter(b, a, demod2)
        exc1_hilbert = np.imag(hilbert(exc1))
        exc2_hilbert = np.imag(hilbert(exc2))
        demod1 = input_signal * exc1_hilbert
        demod2 = input_signal * exc2_hilbert
        if detrend:
            b, a = butter(filter_order, [fL, Fc] / (Fs / 2))
        else:
            b, a = butter(filter_order, Fc / (Fs / 2))
        sig1y = lfilter(b, a, demod1)
        sig2y = lfilter(b, a, demod2)
        sig1 = np.sqrt(sig1x**2 + sig1y**2)
        sig2 = np.sqrt(sig2x**2 + sig2y**2)
    return sig1, sig2


def run_lockin_detection(phot, start):
    """Run lockin detection to extract the modulated photometry signals"""
    tau = 10
    filter_order = 5
    detrend = False
    full = True
    detector = phot["data"][5, :]
    exc1 = phot["data"][6, :]
    exc2 = phot["data"][7, :]
    sig1, ref = lockin_detection(detector, exc1, exc2, phot["sampling_rate"],
                                  tau=tau, filter_order=filter_order, detrend=detrend, full=full)
    # Cut off the beginning of the signals to match behavioral data
    sig1 = sig1[start:]
    ref = ref[start:]
    return {"sig1": sig1, "ref": ref}


def whittaker_smooth(data, binary_mask, lambda_):
    data_matrix = np.array(data)
    data_matrix = np.expand_dims(data_matrix, axis=0)
    data_size = data_matrix.size
    identity_matrix = eye(data_size, format="csc")
    diff_matrix = identity_matrix[1:] - identity_matrix[:-1]
    diagonal_matrix = diags(binary_mask, 0, shape=(data_size, data_size))
    A = csc_matrix(diagonal_matrix + (lambda_ * diff_matrix.T * diff_matrix))
    B = csc_matrix(diagonal_matrix * data_matrix.T)
    smoothed_baseline = spsolve(A, B)
    return np.array(smoothed_baseline)


def airPLS(data, lambda_=1e8, max_iterations=50):
    num_data_points = data.shape[0]
    weights = np.ones(num_data_points)
    for i in range(1, max_iterations + 1):
        baseline = whittaker_smooth(data, weights, lambda_)
        delta = data - baseline
        sum_of_neg_deltas = np.abs(delta[delta < 0].sum())
        if sum_of_neg_deltas < 0.001 * (abs(data)).sum() or i == max_iterations:
            if i == max_iterations:
                print(f"WARNING: airPLS reached max iterations ({max_iterations}) before convergence!")
            break
        weights[delta >= 0] = 0
        weights[delta < 0] = np.exp(i * np.abs(delta[delta < 0]) / sum_of_neg_deltas)
        weights[0] = np.exp(i * (delta[delta < 0]).max() / sum_of_neg_deltas)
        weights[-1] = weights[0]
    return baseline


def import_ppd(ppd_file_path):
    with open(ppd_file_path, "rb") as f:
        header_size = int.from_bytes(f.read(2), "little")
        data_header = f.read(header_size)
        data = np.frombuffer(f.read(), dtype=np.dtype("<u2"))
    header_dict = json.loads(data_header)
    volts_per_division = header_dict["volts_per_division"]
    sampling_rate = header_dict["sampling_rate"]
    analog = data >> 1
    digital = ((data & 1) == 1).astype(int)
    if "n_analog_signals" in header_dict.keys():
        n_analog_signals = header_dict["n_analog_signals"]
        n_digital_signals = header_dict["n_digital_signals"]
    else:
        n_analog_signals = 2
        n_digital_signals = 2
    analog_1 = analog[::n_analog_signals] * volts_per_division[0]
    analog_2 = analog[1::n_analog_signals] * volts_per_division[1]
    analog_3 = analog[2::n_analog_signals] * volts_per_division[0] if n_analog_signals == 3 else None
    digital_1 = digital[::n_analog_signals]
    digital_2 = digital[1::n_analog_signals] if n_digital_signals == 2 else None
    pulse_inds_1 = 1 + np.where(np.diff(digital_1) == 1)[0]
    pulse_inds_2 = 1 + np.where(np.diff(digital_2) == 1)[0] if n_digital_signals == 2 else None
    pulse_times_1 = pulse_inds_1 * 1000 / sampling_rate
    pulse_times_2 = pulse_inds_2 * 1000 / sampling_rate if n_digital_signals == 2 else None
    time = np.arange(analog_1.shape[0]) * 1000 / sampling_rate
    data_dict = {
        "filename": os.path.basename(ppd_file_path),
        "analog_1": analog_1,
        "analog_2": analog_2,
        "digital_1": digital_1,
        "digital_2": digital_2,
        "pulse_inds_1": pulse_inds_1,
        "pulse_inds_2": pulse_inds_2,
        "pulse_times_1": pulse_times_1,
        "pulse_times_2": pulse_times_2,
        "time": time,
    }
    if n_analog_signals == 3:
        data_dict.update({"analog_3": analog_3})
    data_dict.update(header_dict)
    return data_dict

---
## Section 1: LabVIEW dLight Isosbestic Pipeline (from raw .phot/.box)

Processing steps: raw → lock-in detection → downsample → rolling mean → airPLS baseline → subtract → median z-score → Lasso fit → dF/F

In [7]:
# Load raw LabVIEW data
test_data_dir = TEST_DATA_DIR / "IM-1478" / "07252022"
phot_file_path = test_data_dir / "IM-1478_2022-07-25_15-24-22____Tim_Conditioning.phot"
box_file_path = test_data_dir / "IM-1478_2022-07-25_15-24-22____Tim_Conditioning.box"

assert phot_file_path.exists(), f"Missing .phot file: {phot_file_path}"
assert box_file_path.exists(), f"Missing .box file: {box_file_path}"

phot = read_phot_data(phot_file_path)
box = read_box_data(box_file_path)
print(f"Phot sampling rate: {phot['sampling_rate']} Hz")
print(f"Phot data shape: {phot['data'].shape}")
print(f"Box data shape: {box['data'].shape}")


# Step 1: Extract port visits and alignment start point
visits_10khz, start = process_pulses(box)
print(f"Start sample (alignment): {start}")
print(f"Number of port visits: {len(visits_10khz)}")


# Step 2: Lock-in detection at 10 kHz
signals = run_lockin_detection(phot, start)
sig1_10khz = signals["sig1"]
ref_10khz = signals["ref"]
print(f"sig1 (470nm) at 10kHz: {sig1_10khz.shape}")
print(f"ref (405nm) at 10kHz: {ref_10khz.shape}")


# Step 3: Downsample from 10 kHz to 250 Hz
SR = 10000  # Original sampling rate
Fs = 250    # Target sampling rate
downsample_factor = int(SR / Fs)

raw_green = pd.Series(np.squeeze(sig1_10khz)[::downsample_factor])
raw_reference = pd.Series(np.squeeze(ref_10khz)[::downsample_factor])
port_visits = np.divide(visits_10khz, downsample_factor).astype(int)

print(f"Downsampled to {Fs} Hz")
print(f"raw_green: {raw_green.shape}")
print(f"raw_reference: {raw_reference.shape}")
print(f"port_visits: {port_visits.shape}")


# Step 4: Rolling mean smoothing
smooth_window = int(Fs / 30)
min_periods = 1
print(f"Smoothing with rolling mean, window={smooth_window}, min_periods={min_periods}")

smoothed_reference = np.array(
    raw_reference.rolling(window=smooth_window, min_periods=min_periods).mean()
).reshape(len(raw_reference), 1)
smoothed_green = np.array(
    raw_green.rolling(window=smooth_window, min_periods=min_periods).mean()
).reshape(len(raw_green), 1)

print(f"smoothed_green: {smoothed_green.shape}")
print(f"smoothed_reference: {smoothed_reference.shape}")


# Step 5: airPLS baseline estimation (computed on raw signals)
lam = 1e8
max_iter = 50
print(f"Computing airPLS baseline with lambda={lam}, max_iterations={max_iter}")

ref_baseline = airPLS(data=raw_reference.T, lambda_=lam, max_iterations=max_iter).reshape(
    len(raw_reference), 1
)
green_baseline = airPLS(data=raw_green.T, lambda_=lam, max_iterations=max_iter).reshape(
    len(raw_green), 1
)

print(f"green_baseline: {green_baseline.shape}")
print(f"ref_baseline: {ref_baseline.shape}")


# Step 6: Subtract baselines from smoothed signals
baseline_subtracted_ref = smoothed_reference - ref_baseline
baseline_subtracted_green = smoothed_green - green_baseline

print(f"baseline_subtracted_green: {baseline_subtracted_green.shape}")
print(f"baseline_subtracted_ref: {baseline_subtracted_ref.shape}")


# Step 7: Median z-score normalization
z_scored_reference = (baseline_subtracted_ref - np.median(baseline_subtracted_ref)) / np.std(
    baseline_subtracted_ref
)
z_scored_green = (baseline_subtracted_green - np.median(baseline_subtracted_green)) / np.std(
    baseline_subtracted_green
)

print(f"z_scored_green: {z_scored_green.shape}, mean={z_scored_green.mean():.4f}, std={z_scored_green.std():.4f}")
print(f"z_scored_reference: {z_scored_reference.shape}, mean={z_scored_reference.mean():.4f}, std={z_scored_reference.std():.4f}")


# Step 8: Lasso regression (isosbestic correction)
alpha = 0.0001
print(f"Fitting Lasso regression with alpha={alpha}")

lin = Lasso(alpha=alpha, precompute=True, max_iter=1000, positive=True, random_state=9999, selection="random")
lin.fit(z_scored_reference, z_scored_green)
z_scored_reference_fitted = lin.predict(z_scored_reference).reshape(len(z_scored_reference), 1)

print(f"z_scored_reference_fitted: {z_scored_reference_fitted.shape}")
print(f"Lasso coef: {lin.coef_}, intercept: {lin.intercept_}")


# Step 9: dF/F (final corrected signal)
z_scored_green_dFF = z_scored_green - z_scored_reference_fitted

print(f"z_scored_green_dFF: {z_scored_green_dFF.shape}")
print(f"  mean={z_scored_green_dFF.mean():.4f}, std={z_scored_green_dFF.std():.4f}")

Phot sampling rate: 10000 Hz
Phot data shape: (8, 52626000)
Box data shape: (3, 52626000)
Start sample (alignment): 578116
Number of port visits: 188
sig1 (470nm) at 10kHz: (52047884,)
ref (405nm) at 10kHz: (52047884,)
Downsampled to 250 Hz
raw_green: (1301198,)
raw_reference: (1301198,)
port_visits: (188,)
Smoothing with rolling mean, window=8, min_periods=1
smoothed_green: (1301198, 1)
smoothed_reference: (1301198, 1)
Computing airPLS baseline with lambda=100000000.0, max_iterations=50
green_baseline: (1301198, 1)
ref_baseline: (1301198, 1)
baseline_subtracted_green: (1301198, 1)
baseline_subtracted_ref: (1301198, 1)
z_scored_green: (1301198, 1), mean=0.2069, std=1.0000
z_scored_reference: (1301198, 1), mean=0.0284, std=1.0000
Fitting Lasso regression with alpha=0.0001
z_scored_reference_fitted: (1301198, 1)
Lasso coef: [0.1837755], intercept: [0.20170041]
z_scored_green_dFF: (1301198, 1)
  mean=0.0000, std=0.9829


In [8]:
 # Spot-check: compare final output to existing reference CSV
reference_csv_path = test_data_dir / "IM-1478_07252022_h_sampleframe.csv"
if reference_csv_path.exists():
    ref_df = pd.read_csv(reference_csv_path)
    ref_green = ref_df["green"].values
    # The old pipeline from signals.mat produces data that matches the CSV.
    # From raw files, there may be extra samples at the front due to alignment differences.
    our_dFF = z_scored_green_dFF.flatten()
    offset = len(our_dFF) - len(ref_green)
    print(f"Our dFF length: {len(our_dFF)}, Reference length: {len(ref_green)}, Offset: {offset}")
    if offset >= 0:
        trimmed = our_dFF[offset:]
        print(f"  max diff = {np.max(np.abs(trimmed - ref_green)):.2e}")
        print(f"  correlation = {np.corrcoef(trimmed, ref_green)[0, 1]:.6f}")
else:
    print(f"Reference CSV not found at {reference_csv_path}, skipping spot-check")

Our dFF length: 1301198, Reference length: 1301198, Offset: 0
  max diff = 7.32e-03
  correlation = 1.000000


In [9]:
# Save LabVIEW dLight intermediates
# Note: sig1_10khz and ref_10khz are NOT saved — they are 833 MB of 10 kHz data not used by any test.
labview_output_path = OUTPUT_DIR / "labview_dlight_intermediates.npz"

np.savez_compressed(
    labview_output_path,
    # Downsampled raw signals (250 Hz)
    raw_green=raw_green.values,
    raw_reference=raw_reference.values,
    # After rolling mean smoothing
    smoothed_green=smoothed_green,
    smoothed_reference=smoothed_reference,
    # airPLS baselines (computed on raw)
    green_baseline=green_baseline,
    ref_baseline=ref_baseline,
    # After baseline subtraction (smoothed - baseline)
    baseline_subtracted_green=baseline_subtracted_green,
    baseline_subtracted_ref=baseline_subtracted_ref,
    # After median z-score
    z_scored_green=z_scored_green,
    z_scored_reference=z_scored_reference,
    # After Lasso fit
    z_scored_reference_fitted=z_scored_reference_fitted,
    # Final dF/F
    z_scored_green_dFF=z_scored_green_dFF,
    # Port visits (in 250 Hz sample indices)
    port_visits=port_visits,
    # Metadata
    sampling_rate=np.array(Fs),
    original_sampling_rate=np.array(SR),
    smooth_window=np.array(smooth_window),
    min_periods=np.array(min_periods),
    airpls_lambda=np.array(lam),
    airpls_max_iterations=np.array(max_iter),
    lasso_alpha=np.array(alpha),
)

print(f"Saved LabVIEW dLight intermediates to {labview_output_path}")
print(f"File size: {labview_output_path.stat().st_size / 1e6:.1f} MB")

Saved LabVIEW dLight intermediates to /Users/steph/berkelab/jdb_to_nwb/tests/test_data/downloaded/reference/1c415f98146fea2f0f9e2d9bf39c442569048fa3/labview_dlight_intermediates.npz
File size: 105.8 MB


---
## Section 2: LabVIEW dLight Isosbestic Pipeline (from signals.mat)

Same processing as Section 1 but starting from the pre-demodulated `signals.mat` file (skips lock-in detection).
This is the path used by `test_add_photometry_from_signals_mat`.

In [10]:
# Load signals.mat
mat_file_path = test_data_dir / "signals.mat"
assert mat_file_path.exists(), f"Missing signals.mat: {mat_file_path}"

signals_mat = scipy.io.loadmat(mat_file_path, matlab_compatible=True)
print(f"signals.mat keys: {[k for k in signals_mat.keys() if not k.startswith('__')]}")

# Downsample from 10 kHz to 250 Hz (same as Section 1 but starting from .mat)
SR_mat = 10000
Fs_mat = 250
downsample_factor_mat = int(SR_mat / Fs_mat)

mat_raw_green = pd.Series(np.squeeze(signals_mat["sig1"])[::downsample_factor_mat])
mat_raw_reference = pd.Series(np.squeeze(signals_mat["ref"])[::downsample_factor_mat])
mat_port_visits = np.divide(np.squeeze(signals_mat["visits"]), downsample_factor_mat).astype(int)

print(f"mat_raw_green: {mat_raw_green.shape}")
print(f"mat_raw_reference: {mat_raw_reference.shape}")
print(f"mat_port_visits: {mat_port_visits.shape}")

# Rolling mean smoothing
mat_smooth_window = int(Fs_mat / 30)
mat_min_periods = 1

mat_smoothed_reference = np.array(
    mat_raw_reference.rolling(window=mat_smooth_window, min_periods=mat_min_periods).mean()
).reshape(len(mat_raw_reference), 1)
mat_smoothed_green = np.array(
    mat_raw_green.rolling(window=mat_smooth_window, min_periods=mat_min_periods).mean()
).reshape(len(mat_raw_green), 1)

# airPLS baseline estimation (computed on RAW signals)
mat_lam = 1e8
mat_max_iter = 50
print(f"Computing airPLS baseline with lambda={mat_lam}, max_iterations={mat_max_iter}")

mat_ref_baseline = airPLS(data=mat_raw_reference.T, lambda_=mat_lam, max_iterations=mat_max_iter).reshape(
    len(mat_raw_reference), 1
)
mat_green_baseline = airPLS(data=mat_raw_green.T, lambda_=mat_lam, max_iterations=mat_max_iter).reshape(
    len(mat_raw_green), 1
)

# Subtract baselines from smoothed signals
mat_baseline_subtracted_ref = mat_smoothed_reference - mat_ref_baseline
mat_baseline_subtracted_green = mat_smoothed_green - mat_green_baseline

# Median z-score normalization
mat_z_scored_reference = (mat_baseline_subtracted_ref - np.median(mat_baseline_subtracted_ref)) / np.std(
    mat_baseline_subtracted_ref
)
mat_z_scored_green = (mat_baseline_subtracted_green - np.median(mat_baseline_subtracted_green)) / np.std(
    mat_baseline_subtracted_green
)

# Lasso regression (isosbestic correction)
mat_alpha = 0.0001
mat_lin = Lasso(alpha=mat_alpha, precompute=True, max_iter=1000, positive=True, random_state=9999, selection="random")
mat_lin.fit(mat_z_scored_reference, mat_z_scored_green)
mat_z_scored_reference_fitted = mat_lin.predict(mat_z_scored_reference).reshape(len(mat_z_scored_reference), 1)

# dF/F
mat_z_scored_green_dFF = mat_z_scored_green - mat_z_scored_reference_fitted

print(f"mat_z_scored_green_dFF: {mat_z_scored_green_dFF.shape}")
print(f"  mean={mat_z_scored_green_dFF.mean():.4f}, std={mat_z_scored_green_dFF.std():.4f}")

signals.mat keys: ['loc', 'ref', 'sig1', 'sig2', 'visits']
mat_raw_green: (1301198,)
mat_raw_reference: (1301198,)
mat_port_visits: (188,)
Computing airPLS baseline with lambda=100000000.0, max_iterations=50
mat_z_scored_green_dFF: (1301198, 1)
  mean=-0.0000, std=0.9829


In [11]:
# Spot-check: compare to reference CSV (should match closely since signals.mat was used to generate it)
reference_csv_path = test_data_dir / "IM-1478_07252022_h_sampleframe.csv"
if reference_csv_path.exists():
    ref_df = pd.read_csv(reference_csv_path)
    ref_green = ref_df["green"].values
    our_dFF = mat_z_scored_green_dFF.flatten()
    offset = len(our_dFF) - len(ref_green)
    print(f"Our dFF length: {len(our_dFF)}, Reference length: {len(ref_green)}, Offset: {offset}")
    if offset >= 0:
        trimmed = our_dFF[offset:]
        print(f"  max diff = {np.max(np.abs(trimmed - ref_green)):.2e}")
        print(f"  correlation = {np.corrcoef(trimmed, ref_green)[0, 1]:.6f}")
else:
    print(f"Reference CSV not found at {reference_csv_path}, skipping spot-check")

Our dFF length: 1301198, Reference length: 1301198, Offset: 0
  max diff = 7.32e-03
  correlation = 1.000000


In [12]:
# Save signals.mat dLight intermediates
mat_output_path = OUTPUT_DIR / "labview_mat_dlight_intermediates.npz"

np.savez_compressed(
    mat_output_path,
    # Downsampled raw signals (250 Hz)
    raw_green=mat_raw_green.values,
    raw_reference=mat_raw_reference.values,
    # After rolling mean smoothing
    smoothed_green=mat_smoothed_green,
    smoothed_reference=mat_smoothed_reference,
    # airPLS baselines (computed on raw)
    green_baseline=mat_green_baseline,
    ref_baseline=mat_ref_baseline,
    # After baseline subtraction (smoothed - baseline)
    baseline_subtracted_green=mat_baseline_subtracted_green,
    baseline_subtracted_ref=mat_baseline_subtracted_ref,
    # After median z-score
    z_scored_green=mat_z_scored_green,
    z_scored_reference=mat_z_scored_reference,
    # After Lasso fit
    z_scored_reference_fitted=mat_z_scored_reference_fitted,
    # Final dF/F
    z_scored_green_dFF=mat_z_scored_green_dFF,
    # Port visits (in 250 Hz sample indices)
    port_visits=mat_port_visits,
    # Metadata
    sampling_rate=np.array(Fs_mat),
    original_sampling_rate=np.array(SR_mat),
    smooth_window=np.array(mat_smooth_window),
    min_periods=np.array(mat_min_periods),
    airpls_lambda=np.array(mat_lam),
    airpls_max_iterations=np.array(mat_max_iter),
    lasso_alpha=np.array(mat_alpha),
)

print(f"Saved signals.mat dLight intermediates to {mat_output_path}")
print(f"File size: {mat_output_path.stat().st_size / 1e6:.1f} MB")

Saved signals.mat dLight intermediates to /Users/steph/berkelab/jdb_to_nwb/tests/test_data/downloaded/reference/1c415f98146fea2f0f9e2d9bf39c442569048fa3/labview_mat_dlight_intermediates.npz
File size: 105.8 MB


---
## Section 3: pyPhotometry gACh4h + rDA3m Ratiometric Pipeline

Processing steps: raw → compute ratio → lowpass 10Hz → highpass 0.001Hz → mean z-score

In [13]:
# Load pyPhotometry data
ppd_test_data_dir = TEST_DATA_DIR / "IM-1770_corvette" / "11062024"
ppd_file_path = ppd_test_data_dir / "Lhem_barswitch_GACh4h_rDA3m_CKTL-2024-11-06-185407.ppd"

assert ppd_file_path.exists(), f"Missing .ppd file: {ppd_file_path}"

ppd_data = import_ppd(ppd_file_path)
sampling_rate = ppd_data['sampling_rate']
print(f"Sampling rate: {sampling_rate} Hz")
print(f"Signals present: analog_1, analog_2" + (", analog_3" if ppd_data.get('analog_3') is not None else ""))


# Step 1: Extract raw signals
# Jose's setup: analog_1=470nm (gACh4h), analog_2=565nm (rDA3m), analog_3=405nm
ppd_raw_green = pd.Series(ppd_data['analog_1'])    # 470nm
ppd_raw_red = pd.Series(ppd_data['analog_2'])      # 565nm
ppd_raw_405 = pd.Series(ppd_data['analog_3'])      # 405nm
ppd_raw_ratio = ppd_raw_green / ppd_raw_405        # 470/405 ratio

# Port visits
ppd_visits = ppd_data['pulse_inds_1'][1:]  # Skip first pulse (trigger)

print(f"raw_green (470nm): {ppd_raw_green.shape}")
print(f"raw_red (565nm): {ppd_raw_red.shape}")
print(f"raw_405: {ppd_raw_405.shape}")
print(f"raw_ratio (470/405): {ppd_raw_ratio.shape}")
print(f"port_visits: {ppd_visits.shape}")


# Step 2: Lowpass filter at 10 Hz (Butterworth order 2)
lowpass_cutoff = 10
filter_order = 2
print(f"Lowpass filter: cutoff={lowpass_cutoff} Hz, order={filter_order}")

b, a = butter(filter_order, lowpass_cutoff, btype='low', fs=sampling_rate)
ppd_green_lowpass = filtfilt(b, a, ppd_raw_green)
ppd_red_lowpass = filtfilt(b, a, ppd_raw_red)
ppd_ratio_lowpass = filtfilt(b, a, ppd_raw_ratio)
ppd_lowpass_405 = filtfilt(b, a, ppd_raw_405)

print(f"green_lowpass: {ppd_green_lowpass.shape}")
print(f"red_lowpass: {ppd_red_lowpass.shape}")


# Step 3: Highpass filter at 0.001 Hz (Butterworth order 2)
highpass_cutoff = 0.001
print(f"Highpass filter: cutoff={highpass_cutoff} Hz, order={filter_order}")

b, a = butter(filter_order, highpass_cutoff, btype='high', fs=sampling_rate)
ppd_green_highpass = filtfilt(b, a, ppd_green_lowpass, padtype='even')
ppd_red_highpass = filtfilt(b, a, ppd_red_lowpass, padtype='even')
ppd_ratio_highpass = filtfilt(b, a, ppd_ratio_lowpass, padtype='even')
ppd_highpass_405 = filtfilt(b, a, ppd_lowpass_405, padtype='even')

print(f"green_highpass: {ppd_green_highpass.shape}")
print(f"red_highpass: {ppd_red_highpass.shape}")


# Step 4: Mean z-score normalization
ppd_green_zscored = (ppd_green_highpass - ppd_green_highpass.mean()) / ppd_green_highpass.std()
ppd_red_zscored = (ppd_red_highpass - ppd_red_highpass.mean()) / ppd_red_highpass.std()
ppd_zscored_405 = (ppd_highpass_405 - ppd_highpass_405.mean()) / ppd_highpass_405.std()
ppd_ratio_zscored = (ppd_ratio_highpass - ppd_ratio_highpass.mean()) / ppd_ratio_highpass.std()

print(f"green_zscored: mean={ppd_green_zscored.mean():.6f}, std={ppd_green_zscored.std():.6f}")
print(f"red_zscored: mean={ppd_red_zscored.mean():.6f}, std={ppd_red_zscored.std():.6f}")
print(f"ratio_zscored: mean={ppd_ratio_zscored.mean():.6f}, std={ppd_ratio_zscored.std():.6f}")
print(f"zscored_405: mean={ppd_zscored_405.mean():.6f}, std={ppd_zscored_405.std():.6f}")

Sampling rate: 86 Hz
Signals present: analog_1, analog_2, analog_3
raw_green (470nm): (624747,)
raw_red (565nm): (624747,)
raw_405: (624747,)
raw_ratio (470/405): (624747,)
port_visits: (220,)
Lowpass filter: cutoff=10 Hz, order=2
green_lowpass: (624747,)
red_lowpass: (624747,)
Highpass filter: cutoff=0.001 Hz, order=2
green_highpass: (624747,)
red_highpass: (624747,)
green_zscored: mean=0.000000, std=1.000000
red_zscored: mean=0.000000, std=1.000000
ratio_zscored: mean=-0.000000, std=1.000000
zscored_405: mean=-0.000000, std=1.000000


In [14]:
# Spot-check: compare to existing reference CSV
ppd_reference_csv_path = ppd_test_data_dir / "sampleframe.csv"
if ppd_reference_csv_path.exists():
    ppd_ref_df = pd.read_csv(ppd_reference_csv_path)
    print("Comparing to reference sampleframe.csv:")
    for our, ref_col in [
        (ppd_raw_green.values, "raw_green"),
        (ppd_green_zscored, "green_z_scored"),
        (ppd_red_zscored, "red_z_scored"),
        (ppd_ratio_zscored, "ratio_z_scored"),
    ]:
        ref = ppd_ref_df[ref_col].values
        print(f"  {ref_col}: max diff = {np.max(np.abs(our - ref)):.2e}, correlation = {np.corrcoef(our, ref)[0, 1]:.6f}")
else:
    print(f"Reference CSV not found at {ppd_reference_csv_path}, skipping spot-check")

Comparing to reference sampleframe.csv:
  raw_green: max diff = 8.33e-17, correlation = 1.000000
  green_z_scored: max diff = 5.54e-06, correlation = 1.000000
  red_z_scored: max diff = 8.15e-06, correlation = 1.000000
  ratio_z_scored: max diff = 4.88e-06, correlation = 1.000000


In [15]:
# Save pyPhotometry gACh4h + rDA3m intermediates
ppd_output_path = OUTPUT_DIR / "pyphotometry_gach_rda_intermediates.npz"

np.savez_compressed(
    ppd_output_path,
    # Raw signals
    raw_green=ppd_raw_green.values,
    raw_red=ppd_raw_red.values,
    raw_405=ppd_raw_405.values,
    raw_ratio=ppd_raw_ratio.values,
    # After lowpass filter
    green_lowpass=ppd_green_lowpass,
    red_lowpass=ppd_red_lowpass,
    ratio_lowpass=ppd_ratio_lowpass,
    lowpass_405=ppd_lowpass_405,
    # After highpass filter
    green_highpass=ppd_green_highpass,
    red_highpass=ppd_red_highpass,
    ratio_highpass=ppd_ratio_highpass,
    highpass_405=ppd_highpass_405,
    # After mean z-score
    green_zscored=ppd_green_zscored,
    red_zscored=ppd_red_zscored,
    ratio_zscored=ppd_ratio_zscored,
    zscored_405=ppd_zscored_405,
    # Port visits
    port_visits=ppd_visits,
    # Metadata
    sampling_rate=np.array(sampling_rate),
    lowpass_cutoff=np.array(lowpass_cutoff),
    highpass_cutoff=np.array(highpass_cutoff),
    filter_order=np.array(filter_order),
)

print(f"Saved pyPhotometry intermediates to {ppd_output_path}")
print(f"File size: {ppd_output_path.stat().st_size / 1e6:.1f} MB")

Saved pyPhotometry intermediates to /Users/steph/berkelab/jdb_to_nwb/tests/test_data/downloaded/reference/1c415f98146fea2f0f9e2d9bf39c442569048fa3/pyphotometry_gach_rda_intermediates.npz
File size: 59.1 MB


---
## Section 4: pyPhotometry 2-signal dLight Isosbestic Pipeline

This is the 2-signal pyPhotometry case (dLight, analog_1=470nm, analog_2=405nm). On 1c415f98146fea2f0f9e2d9bf39c442569048fa3, this goes through `process_and_add_pyphotometry_to_nwb` → `process_and_add_ys_photometry_to_nwb`, using the same rolling mean + airPLS + median z-score + Lasso isosbestic correction pipeline as LabVIEW.

In [16]:
# Load 2-signal pyPhotometry data
ppd2_test_data_dir = TEST_DATA_DIR / "IM-1947" / "20260422"
ppd2_file_path = ppd2_test_data_dir / "IM-1947_L-2026-04-22-145706.ppd"

assert ppd2_file_path.exists(), f"Missing .ppd file: {ppd2_file_path}"
ppd2_data = import_ppd(ppd2_file_path)
ppd2_sampling_rate = ppd2_data['sampling_rate']
print(f"Sampling rate: {ppd2_sampling_rate} Hz")
print(f"Signals present: analog_1, analog_2" + (", analog_3" if ppd2_data.get('analog_3') is not None else ""))
assert ppd2_data.get('analog_3') is None, "Expected 2-signal .ppd file but got 3 signals!"


# Step 1: Extract raw signals (2-signal: analog_1=470nm dLight, analog_2=405nm isosbestic)
ppd2_raw_green = pd.Series(ppd2_data['analog_1'])    # 470nm
ppd2_raw_reference = pd.Series(ppd2_data['analog_2'])  # 405nm
ppd2_visits = ppd2_data['pulse_inds_1'][1:]  # Skip first pulse (trigger)

print(f"raw_green (470nm): {ppd2_raw_green.shape}")
print(f"raw_reference (405nm): {ppd2_raw_reference.shape}")
print(f"port_visits: {ppd2_visits.shape}")


# Step 2: Rolling mean smoothing (same as LabVIEW isosbestic pipeline)
ppd2_smooth_window = int(ppd2_sampling_rate / 30)
ppd2_min_periods = 1
print(f"\nSmoothing with rolling mean, window={ppd2_smooth_window}, min_periods={ppd2_min_periods}")

ppd2_smoothed_reference = np.array(
    ppd2_raw_reference.rolling(window=ppd2_smooth_window, min_periods=ppd2_min_periods).mean()
).reshape(len(ppd2_raw_reference), 1)
ppd2_smoothed_green = np.array(
    ppd2_raw_green.rolling(window=ppd2_smooth_window, min_periods=ppd2_min_periods).mean()
).reshape(len(ppd2_raw_green), 1)


# Step 3: airPLS baseline estimation (computed on RAW signals)
ppd2_lam = 1e8
ppd2_max_iter = 50
print(f"Computing airPLS baseline with lambda={ppd2_lam}, max_iterations={ppd2_max_iter}")

ppd2_ref_baseline = airPLS(data=ppd2_raw_reference.T, lambda_=ppd2_lam, max_iterations=ppd2_max_iter).reshape(len(ppd2_raw_reference), 1)
ppd2_green_baseline = airPLS(data=ppd2_raw_green.T, lambda_=ppd2_lam, max_iterations=ppd2_max_iter).reshape(len(ppd2_raw_green), 1)


# Step 4: Subtract baselines from smoothed signals
ppd2_baseline_subtracted_ref = ppd2_smoothed_reference - ppd2_ref_baseline
ppd2_baseline_subtracted_green = ppd2_smoothed_green - ppd2_green_baseline


# Step 5: Median z-score normalization
ppd2_z_scored_reference = (ppd2_baseline_subtracted_ref - np.median(ppd2_baseline_subtracted_ref)) / np.std(ppd2_baseline_subtracted_ref)
ppd2_z_scored_green = (ppd2_baseline_subtracted_green - np.median(ppd2_baseline_subtracted_green)) / np.std(ppd2_baseline_subtracted_green)

print(f"z_scored_green: mean={ppd2_z_scored_green.mean():.4f}, std={ppd2_z_scored_green.std():.4f}")
print(f"z_scored_reference: mean={ppd2_z_scored_reference.mean():.4f}, std={ppd2_z_scored_reference.std():.4f}")


# Step 6: Lasso regression (isosbestic correction)
ppd2_alpha = 0.0001
print(f"\nFitting Lasso regression with alpha={ppd2_alpha}")

ppd2_lin = Lasso(alpha=ppd2_alpha, precompute=True, max_iter=1000, positive=True, random_state=9999, selection="random")
ppd2_lin.fit(ppd2_z_scored_reference, ppd2_z_scored_green)
ppd2_z_scored_reference_fitted = ppd2_lin.predict(ppd2_z_scored_reference).reshape(len(ppd2_z_scored_reference), 1)

print(f"Lasso coef: {ppd2_lin.coef_}, intercept: {ppd2_lin.intercept_}")


# Step 7: dF/F
ppd2_z_scored_green_dFF = ppd2_z_scored_green - ppd2_z_scored_reference_fitted
print(f"\nppd2_z_scored_green_dFF: {ppd2_z_scored_green_dFF.shape}")
print(f"  mean={ppd2_z_scored_green_dFF.mean():.4f}, std={ppd2_z_scored_green_dFF.std():.4f}")

Sampling rate: 130 Hz
Signals present: analog_1, analog_2
raw_green (470nm): (753251,)
raw_reference (405nm): (753251,)
port_visits: (373,)

Smoothing with rolling mean, window=4, min_periods=1
Computing airPLS baseline with lambda=100000000.0, max_iterations=50
z_scored_green: mean=0.2052, std=1.0000
z_scored_reference: mean=0.1615, std=1.0000

Fitting Lasso regression with alpha=0.0001
Lasso coef: [0.06673237], intercept: [0.19446455]

ppd2_z_scored_green_dFF: (753251, 1)
  mean=-0.0000, std=0.9978


In [17]:
# Save 2-signal pyPhotometry dLight intermediates
ppd2_output_path = OUTPUT_DIR / "pyphotometry_dlight_intermediates.npz"

np.savez_compressed(
    ppd2_output_path,
    # Raw signals
    raw_green=ppd2_raw_green.values,
    raw_reference=ppd2_raw_reference.values,
    # After rolling mean smoothing
    smoothed_green=ppd2_smoothed_green,
    smoothed_reference=ppd2_smoothed_reference,
    # airPLS baselines (computed on raw)
    green_baseline=ppd2_green_baseline,
    ref_baseline=ppd2_ref_baseline,
    # After baseline subtraction (smoothed - baseline)
    baseline_subtracted_green=ppd2_baseline_subtracted_green,
    baseline_subtracted_ref=ppd2_baseline_subtracted_ref,
    # After median z-score
    z_scored_green=ppd2_z_scored_green,
    z_scored_reference=ppd2_z_scored_reference,
    # After Lasso fit
    z_scored_reference_fitted=ppd2_z_scored_reference_fitted,
    # Final dF/F
    z_scored_green_dFF=ppd2_z_scored_green_dFF,
    # Port visits
    port_visits=ppd2_visits,
    # Metadata
    sampling_rate=np.array(ppd2_sampling_rate),
    smooth_window=np.array(ppd2_smooth_window),
    min_periods=np.array(ppd2_min_periods),
    airpls_lambda=np.array(ppd2_lam),
    airpls_max_iterations=np.array(ppd2_max_iter),
    lasso_alpha=np.array(ppd2_alpha),
)

print(f"Saved 2-signal pyPhotometry dLight intermediates to {ppd2_output_path}")
print(f"File size: {ppd2_output_path.stat().st_size / 1e6:.1f} MB")

Saved 2-signal pyPhotometry dLight intermediates to /Users/steph/berkelab/jdb_to_nwb/tests/test_data/downloaded/reference/1c415f98146fea2f0f9e2d9bf39c442569048fa3/pyphotometry_dlight_intermediates.npz
File size: 45.0 MB


In [18]:
# Summary
print("\n" + "="*60)
print("Reference data generation complete!")
print("="*60)
print(f"\nOutput files:")
for f in sorted(OUTPUT_DIR.glob("*.npz")):
    data = np.load(f)
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
    print(f"    Keys: {list(data.keys())}")
    data.close()
print(f"\nGenerated from main commit: {COMMIT_TAG}")


Reference data generation complete!

Output files:
  labview_dlight_intermediates.npz (105.8 MB)
    Keys: ['raw_green', 'raw_reference', 'smoothed_green', 'smoothed_reference', 'green_baseline', 'ref_baseline', 'baseline_subtracted_green', 'baseline_subtracted_ref', 'z_scored_green', 'z_scored_reference', 'z_scored_reference_fitted', 'z_scored_green_dFF', 'port_visits', 'sampling_rate', 'original_sampling_rate', 'smooth_window', 'min_periods', 'airpls_lambda', 'airpls_max_iterations', 'lasso_alpha']
  labview_mat_dlight_intermediates.npz (105.8 MB)
    Keys: ['raw_green', 'raw_reference', 'smoothed_green', 'smoothed_reference', 'green_baseline', 'ref_baseline', 'baseline_subtracted_green', 'baseline_subtracted_ref', 'z_scored_green', 'z_scored_reference', 'z_scored_reference_fitted', 'z_scored_green_dFF', 'port_visits', 'sampling_rate', 'original_sampling_rate', 'smooth_window', 'min_periods', 'airpls_lambda', 'airpls_max_iterations', 'lasso_alpha']
  pyphotometry_dlight_intermediate